In [1]:
# import dependencies
import pandas as pd
import re
import numpy as np
from rapidfuzz import process

### Load csv's

In [2]:
# load acs data (for merging town names, GEOID, population and housing unit data)
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")

Used this [list of HMGP activities](https://www.fema.gov/sites/default/files/2020-08/fema_mt-egrants-guide-to-eligible-activities-and-codes_job_aid_March_2018.pdf) to create a csv to classify projectType.

In [3]:
# load FEMA project codes (for categorizing projects)
codes_df = pd.read_csv("../data/resources/FEMA_project_codes.csv")
codes_df.head()

,code,main_category,subcategory,flood_mitigation_flag
0,100.1,planning,public_awareness,0
1,103.1,planning,feasibility_study,0
2,104.1,planning,codes_standards,0
3,106.1,planning,other_nonconstruction,0
4,106.2,planning,other_nonconstruction,0


In [4]:
# load FEMA Hazard Mitigation Assistance Projects for Vermont (the data of interest)
df = pd.read_csv("../data/raw/funding/vt_hma_awards.csv")
print(df.shape)
df.head()

(756, 33)


,projectIdentifier,programArea,programFy,region,state,stateNumberCode,county,countyCode,disasterNumber,projectCounties,...,federalShareObligated,subrecipientAdminCostAmt,srmcObligatedAmt,recipientAdminCostAmt,costSharePercentage,benefitCostRatio,netValueBenefits,numberOfFinalProperties,numberOfProperties,id
0,DR-4532-0005-R,HMGP,2020,1,Vermont,50,Rutland,21.0,4532.0,RUTLAND,...,140188.70,0.0,0.0,0.0,0.90,1.13,323000.0,1,1,344986e8-9849-4f9a-9504-436d4a3c92c7
1,DR-4720-0047-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,237540.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,1bce9785-0760-4558-bf79-6cd8047bad9b
2,DR-4720-0069-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,265767.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,62892290-55db-4565-90e1-29436e27138c
3,DR-4720-0043-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,121983.00,0.0,0.0,0.0,0.75,0.00,0.0,1,1,65ee2247-4a87-42b0-a1d0-70c290c0a844
4,DR-4022-0109-R,HMGP,2011,1,Vermont,50,Washington,23.0,4022.0,WASHINGTON,...,0.00,0.0,0.0,0.0,0.75,0.00,0.0,0,1,d3018dab-e76a-4236-8d76-a5da6fc581ff


# Create cleaned projects csv

In [5]:
# prune columns to those needed for analysis
df = df[
    [
        "projectIdentifier",
        "programArea",
        "programFy",
        "disasterNumber",
        "projectCounties",
        "projectType",
        "status",
        "subrecipient",
        "dateApproved",
        "projectAmount",
        "federalShareObligated",
        "costSharePercentage",
        "benefitCostRatio",
        "netValueBenefits",
        "numberOfFinalProperties",
        "numberOfProperties",
    ]
]
df.shape

(756, 16)

### Categorize projects and funding

In [6]:
# merge project codes csv with FEMA HMA dataframe, including handling multiple codes in projectType

# extract all codes from projectType, creating a list of codes for each project
# regex: one or more digits, followed by a period, followed by one or more digits (e.g. 200.1)
df = df.copy()  # avoid SettingWithCopyWarning
df["codes"] = df["projectType"].str.findall(r"\d+\.\d+")

# explode codes so each project-code is a row
df_exploded = df.explode("codes")

# ensure string type for merging
df_exploded["codes"] = df_exploded["codes"].astype(str)
codes_df["code"] = codes_df["code"].astype(str)

# merge exploded df with project type codebook
df_merged = df_exploded.merge(codes_df, left_on="codes", right_on="code", how="left")

# create dummies for main_category and concatenate back to df
category_dummies = pd.get_dummies(df_merged["main_category"])
df_merged = pd.concat([df_merged, category_dummies], axis=1)

# create {main_category}_funding columns for each main_category
df_merged["n_codes"] = df_merged.groupby("projectIdentifier")["codes"].transform(
    "count"
)
# split funding equally across codes for each project, then multiply by category dummies to get funding for each main category
df_merged["allocated_funding"] = (
    df_merged["federalShareObligated"] / df_merged["n_codes"]
)
for col in category_dummies.columns:
    df_merged["funding_" + col] = df_merged[col] * df_merged["allocated_funding"]

# aggregate back to project level,
# with flags for each main category and flood mitigation, and concatenated subcategories
agg = (
    df_merged.groupby("projectIdentifier")
    .agg(
        {
            # for projects with multiple codes, flag if any code has a flood mitigation flag of 1
            "flood_mitigation_flag": "max",
            # join all unique, non-null subcategories for the project
            "subcategory": lambda x: ",".join(sorted(set(x.dropna()))),
            # old approach, superseded by dummies, left for testing
            # "main_category": lambda x: ",".join(sorted(set(x.dropna()))),
            # flag if any code has a 1 for each main category
            **{col: "max" for col in category_dummies.columns},
            # sum the funding for each main category across all codes for the project
            **{"funding_" + col: "sum" for col in category_dummies.columns},
        }
    )
    .reset_index()
)

# merge back to original df
df_coded = df.merge(agg, on="projectIdentifier", how="left")
df = df_coded
print(df.shape)
df.head()

(756, 39)


,projectIdentifier,programArea,programFy,disasterNumber,projectCounties,projectType,status,subrecipient,dateApproved,projectAmount,...,funding_acquisition_buyouts,funding_admin,funding_ecosystem_restoration,funding_equipment,funding_flood_control,funding_infrastructure_protection,funding_infrastructure_stormwater,funding_other,funding_planning,funding_structure_protection
0,DR-4532-0005-R,HMGP,2020,4532.0,RUTLAND,200.1: Acquisition of Private Real Property (S...,Closed,Pittsfield (Town of),2025-01-13T00:00:00.000Z,155765.23,...,140188.70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DR-4720-0047-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2024-11-14T00:00:00.000Z,316721.00,...,237540.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,DR-4720-0069-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2025-04-21T00:00:00.000Z,354357.00,...,265767.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,DR-4720-0043-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2024-11-14T00:00:00.000Z,162644.00,...,121983.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,DR-4022-0109-R,HMGP,2011,4022.0,WASHINGTON,202.1: Elevation of Private Structures - Riverine,Closed,Waterbury (Town of),2016-05-11T00:00:00.000Z,0.00,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
df["flood_mitigation_flag"].value_counts()

flood_mitigation_flag
1.0    509
0.0    234
Name: count, dtype: int64

### Clean recipients, assign to towns if applicable

In [8]:
# dict for manual town assignments based on subrecipient names
manual_town_dict = {
    # villages inside of towns
    "JEFFERSONVILLE": "CAMBRIDGE TOWN",
    "LYNDONVILLE": "LYNDON TOWN",
    "ENOSBURG FALLS": "ENOSBURGH TOWN",
    # easily identifiable with a certain town
    "RUPERT- HIGHWAY DEPARTMENT": "RUPERT TOWN",
    "STOWE ELECTRIC DEPARTMENT": "STOWE TOWN",
    "MONTPELIER- PUBLIC WORKS DEPARTMENT": "MONTPELIER CITY",
    "BRATTLEBORO HOUSING AUTHORITY": "BRATTLEBORO TOWN",
    "BARRE HISTORICAL SOCIETY INC": "BARRE CITY",
    "ETHAN ALLEN HOMESTEAD TRUST": "BURLINGTON CITY",
    "NORWICH UNIVERSITY": "NORTHFIELD TOWN",
    # changed it's name in 1999, and the solitary project for this subrecipient is from 1996
    "SHERBURNE": "KILLINGTON TOWN",
}


# Barre, Rutland, St. Albans, and Newport have a city and town of the same name
# Vermont is sometimes lacking in creativity
# override function for classifying a few towns with ambiguous names, based on explicit mentions of "TOWN" or "CITY" in the subrecipient name
def classify_barre_rutland_stalbans(subrecipient):
    if pd.isna(subrecipient):
        return None
    s = subrecipient.upper().replace(".", "")
    ambiguous_towns = {
        "BARRE": ["BARRE TOWN", "BARRE CITY", "AMBIGUOUS: BARRE"],
        "RUTLAND": ["RUTLAND TOWN", "RUTLAND CITY", "AMBIGUOUS: RUTLAND"],
        "ST ALBANS": ["ST. ALBANS TOWN", "ST. ALBANS CITY", "AMBIGUOUS: ST. ALBANS"],
        "NEWPORT": ["NEWPORT TOWN", "NEWPORT CITY", "AMBIGUOUS: NEWPORT"],
    }
    for town, [town_name, city_name, ambiguous_name] in ambiguous_towns.items():
        if town in s:
            if "TOWN" in s:
                return town_name
            elif "CITY" in s:
                return city_name
            elif s.strip() == town:
                # default to city (ambiguous name used during data cleaning)
                return city_name
    return None

In [9]:
# regex to clean subrecipient names
def clean_town(x):
    # handle NaN values
    if pd.isna(x):
        return x
    # convert to uppercase
    x = x.upper()
    # remove common prefixes and suffixes
    x = re.sub(r"\(TOWN OF\)|\(CITY OF\)|\(VILLAGE OF\)", "", x)
    x = re.sub(r"TOWN OF|CITY OF|VILLAGE OF", "", x)
    # remove any remaining parentheses
    x = re.sub(r"\(", "", x)
    x = re.sub(r"\)", "", x)
    # remove punctuation, extra whitespace, leading/trailing whitespace
    x = re.sub(r",", "", x)
    x = re.sub(r"\s+", " ", x)
    x = x.strip()
    # remove ' VT' or ' VERMONT' at the end, unless preceded by ' OF '
    # cleans some towns w/o affecting "Univ of Vermont", "State of Vermont", etc.
    # (?<! OF) means "not immediately preceded by ' OF'"
    x = re.sub(r"(?<! OF) (VT|VERMONT)$", "", x)
    return x


# apply cleaning function to subrecipient column to create town_clean
df["town_clean"] = df["subrecipient"].apply(clean_town)

# get and prepare canonical town list from ACS
town_list = acs_df["town_name"].str.upper().str.strip().unique().tolist()
# build dict of base names for structured fuzzy matching of town names that differ only by "TOWN", "CITY", or "VILLAGE"
town_list_base = {}
for t in town_list:
    parts = t.split()
    if len(parts) > 1:
        base = re.sub(r" (TOWN|CITY|VILLAGE)$", "", t)
        town_list_base[base] = t


# function for matching
# checking for exact matches, base name matches, and then fuzzy matches
# uses rapidfuzz for fuzzy matches, with a score cutoff to avoid bad matches
def match_town(name, town_list, score_cutoff=90):
    # handle NaN values
    if pd.isna(name):
        return None
    # check for exact match
    if name in town_list:
        return name
    # check for base match if name ends with " TOWN", " CITY", or " VILLAGE"
    if name in town_list_base:
        return town_list_base[name]
    # if no exact or base match, use fuzzy matching with a score cutoff to avoid bad matches
    # the fuzzy match, in this case, is redundant - exact and base matches catch all cases for Vermont
    match = process.extractOne(name, town_list, score_cutoff=score_cutoff)
    if match:
        return match[0]
    return None


# apply fuzzy matching to create town_match column
df["town_match"] = df["town_clean"].apply(lambda x: match_town(x, town_list))


# handle ambiguous cases, otherwise use fuzzy match
def resolve_town(row):
    # check for explicit classifications of ambiguous cases
    special = classify_barre_rutland_stalbans(row["subrecipient"])
    if special is not None:
        return special
    # then check manual overrides
    cleaned = row["town_clean"]
    if cleaned in manual_town_dict:
        return manual_town_dict[cleaned]
    # then check fuzzy match results
    elif pd.notna(row["town_match"]):
        return row["town_match"]
    else:
        return None


# create final town column using ambiguous case resolution and fuzzy matching
df["town_assigned"] = df.apply(resolve_town, axis=1)
print(df.shape)
df["town_assigned"].value_counts(dropna=False)

(756, 42)


town_assigned
None                271
PAWLET TOWN          16
MONTPELIER CITY      14
RUPERT TOWN          13
CAMBRIDGE TOWN       12
                   ... 
WOODFORD TOWN         1
MORRISTOWN TOWN       1
WALLINGFORD TOWN      1
SHREWSBURY TOWN       1
GUILDHALL TOWN        1
Name: count, Length: 147, dtype: int64

Re: unmatched

Need to resolve ambiguity for Barre / Rutland / St Albans - mostly property acquisition (planning for St Albans) - could merge city and town? but that merges acs variables? - ASSIGNED TO CITY  
Orleans could be a village in Barton (a generator) or the county (a buyout)  
Statewide stuff? Might just drop it. Could distribute per-capita.  
Chittenden, Lamoille, Windham, Windsor, Ottauquechee, could be distributed across the respective county / multiple towns. Could be dropped.  

In [10]:
# review ambiguous and unmatched cases

# # review ambiguous cases (Rutland, Barre, St. Albans, and Newport, which have a city and town of the same name)
# ambig = df[df["town_assigned"].str.startswith("AMBIGUOUS", na=False)]
# print("Ambiguous subrecipients:", ambig[["subrecipient", "town_clean"]])

# # review unmatched cases
# pd.set_option("display.max_rows", None)
# # display where town_assigned is null to check unmatched cases
# unmatched = df[df["town_assigned"].isna()]
# print(
#     f'\n\nNumber of unmatched cases: {len(unmatched[["subrecipient", "town_clean", "town_match", "town_assigned"]].drop_duplicates().sort_values("town_clean"))}'
# )
# unmatched[
#     ["subrecipient", "town_clean", "town_match", "town_assigned"]
# ].drop_duplicates().sort_values("town_clean")

In [11]:
# pd.reset_option("display.max_rows")

In [12]:
# drop columns used for matching but not needed for analysis
df = df.drop(
    [
        "town_clean",
        "town_match",
        "codes",
    ],
    axis=1,
)
df.shape

(756, 39)

### Assign a geographical level and export to csv

In [13]:
# assign a geographic level to each project, based on if a town was assigned and the subrecipient name
def classify_geo_level(row):
    if pd.notna(row.get("town_assigned")):
        return "local"
    s = str(row.get("subrecipient", "")).upper()
    # known county/regional entities in Vermont, based on manual review of subrecipient names
    if "COUNTY" in s or "REGION" in s or "REGIONAL" in s or "COMMISSION" in s:
        return "regional"
    # based on manual review of subrecipient names
    # exclude university from statewide
    if (
        (
            "STATE" in s
            or "VERMONT" in s
            or "VT" in s
            or "PUBLIC" in s
            or "ENVIRONMENTAL" in s
        )
        and "UNIV" not in s
        and "UNIVERSITY" not in s
    ):
        return "statewide"
    return "unknown"


df["geo_level"] = df.apply(classify_geo_level, axis=1)
df.geo_level.value_counts()

geo_level
local        485
statewide    220
unknown       33
regional      18
Name: count, dtype: int64

In [14]:
# export df to csv
df.to_csv("../data/cleaned/fema_hma_projects_clean.csv", index=False)

# Create Town Summary

In [15]:
# check satus counts, before filtering out unapproved projects
df["status"].value_counts()

status
Closed                   527
Approved                 114
Not Approved / Denied     29
Pending                   28
Withdrawn                 25
Obligated                 22
Void                       2
Awarded                    2
Not Selected               1
Completed                  1
Name: count, dtype: int64

In [16]:
# filter to only include approved projects, and projects flagged for flood mitigation
keep_statuses = ["Approved", "Awarded", "Closed", "Completed", "Obligated"]
df_filtered = df[df["status"].isin(keep_statuses)].copy()
df_filtered = df_filtered[df_filtered["flood_mitigation_flag"] == 1]
print(df_filtered["status"].value_counts())

status
Closed       338
Approved      94
Obligated     13
Completed      1
Name: count, dtype: int64


In [17]:
# check to see how many projects exist at the town level after filtering to approved flood mitigation projects
print(df_filtered["geo_level"].value_counts())

geo_level
local        362
statewide     81
unknown        3
Name: count, dtype: int64


In [18]:
# bin projects into funding periods based on programFy, to analyze trends over time
# 2011 and 2023 were major flooding years in Vermont, making natural breakpoints

# convert programFy to numeric (if not already)
df_filtered["programFy"] = pd.to_numeric(df_filtered["programFy"], errors="coerce")


# function to bin years
def funding_period(year):
    if pd.isna(year):
        return "unknown"
    if year < 2011:
        return "pre_2011"
    elif year < 2023:
        return "2011_2022"
    else:
        return "2023_plus"


# apply function to create fiscal_period column and review counts
df_filtered["fiscal_period"] = df_filtered["programFy"].apply(funding_period)
df_filtered["fiscal_period"].value_counts()

fiscal_period
2011_2022    186
pre_2011     166
2023_plus     94
Name: count, dtype: int64

In [19]:
# print total and local spending, before and after filtering to approved flood mitigation projects
total = df["federalShareObligated"].sum()
local = df[df["geo_level"] == "local"]["federalShareObligated"].sum()
print(f"Total federal share obligated: ${total:,.0f}")
print(f"Total federal share obligated to local projects: ${local:,.0f}")
print(local / total)

total_filtered = df_filtered["federalShareObligated"].sum()
local_filtered = df_filtered[df_filtered["geo_level"] == "local"][
    "federalShareObligated"
].sum()
print(f"Total federal share obligated (filtered): ${total_filtered:,.0f}")
print(
    f"Total federal share obligated to local projects (filtered): ${local_filtered:,.0f}"
)
print(local_filtered / total_filtered)

Total federal share obligated: $108,884,329
Total federal share obligated to local projects: $57,889,224
0.5316579893777411
Total federal share obligated (filtered): $97,456,649
Total federal share obligated to local projects (filtered): $56,293,109
0.5776220513751136


In [20]:
# aggregate projects to town level, summing funding and project counts for each town

# fill NaN values in town_assigned with "State, Regional, or Unknown"
# (will have to do again after merging with acs data, below)
df_filtered["town_assigned"] = df_filtered["town_assigned"].fillna(
    "State, Regional, or Unknown"
)

# build aggregation dict for town funding
funding_cols = [col for col in df_filtered.columns if col.startswith("funding_")]
agg_dict = {col: "sum" for col in funding_cols}
agg_dict.update(
    {
        "federalShareObligated": "sum",
        "numberOfFinalProperties": "sum",
        "numberOfProperties": "sum",
    }
)

# aggregate by town_assigned
agg_df = df_filtered.groupby("town_assigned", dropna=False).agg(agg_dict).reset_index()

# check how many towns have funding
print(agg_df.shape)
agg_df.head()

(126, 14)


,town_assigned,funding_acquisition_buyouts,funding_admin,funding_ecosystem_restoration,funding_equipment,funding_flood_control,funding_infrastructure_protection,funding_infrastructure_stormwater,funding_other,funding_planning,funding_structure_protection,federalShareObligated,numberOfFinalProperties,numberOfProperties
0,ANDOVER TOWN,106817.55,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,106817.55,1,1
1,ARLINGTON TOWN,0.00,0.0,0.0,0.0,3125.0,3125.0,0.00,0.0,0.0,3125.0,9375.00,1,1
2,BARNARD TOWN,179837.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,179837.00,1,1
3,BARRE CITY,1039229.57,0.0,0.0,0.0,0.0,0.0,184575.25,0.0,0.0,85369.0,1309173.82,8,10
4,BARRE TOWN,297687.00,0.0,0.0,0.0,0.0,0.0,61132.50,0.0,0.0,0.0,358819.50,3,3


Roughly half of towns have flood mitigation funding

In [21]:
# add funding period columns to town-level agg_df, to analyze trends over time at the town level

# pivot to get funding per period as columns
funding_by_period = (
    df_filtered.pivot_table(
        index="town_assigned",
        columns="fiscal_period",
        values="federalShareObligated",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# merge funding by period back to agg_df
agg_df = agg_df.merge(funding_by_period, on="town_assigned", how="left")

# check that values sum correctly
print(agg_df[["pre_2011", "2011_2022", "2023_plus"]].sum())

# check towns with nonzero funding in each period
print((agg_df[["pre_2011", "2011_2022", "2023_plus"]] > 0).sum())

pre_2011      7971016.65
2011_2022    43279072.88
2023_plus    46206559.27
dtype: float64
pre_2011     78
2011_2022    70
2023_plus    20
dtype: int64


In [22]:
# create cost per property columns
# may be useful? for evaluating property acquisition projects?
agg_df["cost_per_property"] = np.where(
    agg_df["numberOfProperties"] > 0,
    agg_df["federalShareObligated"] / agg_df["numberOfProperties"],
    np.nan,
)
agg_df["cost_per_final_property"] = np.where(
    agg_df["numberOfFinalProperties"] > 0,
    agg_df["federalShareObligated"] / agg_df["numberOfFinalProperties"],
    np.nan,
)

In [23]:
# create funding share columns, to see what proportion of funding went to which category of projects
# may imply town strategies around mitigation, or reliance on buyouts / something else as a mitigation strategy
category_cols = [col for col in agg_df.columns if col.startswith("funding_")]

for col in category_cols:
    share_col = col.replace("funding_", "share_")
    agg_df[share_col] = np.where(
        agg_df["federalShareObligated"] > 0,
        agg_df[col] / agg_df["federalShareObligated"],
        0,
    )

In [24]:
# merge aggregated funding data back to town-level acs data on population and GEOID, to get a complete town-level dataset for analysis
# using an outer join to keep all towns and the Statewide row for reference

# create town_name_upper in acs_df for merging
acs_df["town_name_upper"] = acs_df["town_name"].str.upper().str.strip()

unmatched = agg_df[~agg_df["town_assigned"].isin(acs_df["town_name_upper"])]
print(f'Unmatched towns (should only be State):\n{unmatched["town_assigned"]}')

# outer join to keep all towns, including those without funding,
# AND to keep the Statewide row for reference (will be separated at the end)
town_df = pd.merge(
    acs_df[
        [
            "GEOID",
            "town_name",
            "town_name_upper",
            "total_population",
            "occupied_housing_units",
        ]
    ],
    agg_df,
    left_on="town_name_upper",
    right_on="town_assigned",
    how="outer",
)

# fill remaining NaN values with 0
# numeric_cols = town_df.select_dtypes(include=[np.number]).columns
# town_df[numeric_cols] = town_df[numeric_cols].fillna(0)

# replace NaN in string column with State label, as it wasn't in the ACS data
town_df["town_name"] = town_df["town_name"].replace(
    np.nan, "State, Regional, or Unknown"
)

# drop helper columns
town_df = town_df.drop(columns=["town_name_upper", "town_assigned"])

# checking for 256 towns + 1 row for State, regional, or unknown
town_df.shape

Unmatched towns (should only be State):
102    State, Regional, or Unknown
Name: town_assigned, dtype: object


(257, 32)

In [25]:
# calculate funding per capita and per occupied housing unit, log of those, and add a flag for whether the town received any funding
# (normalize by population to compare across towns of different sizes, and log transform to handle skewness in funding distribution)

# calculate funding per capita and log of funding per capita
town_df["funding_per_capita"] = np.where(
    (town_df["total_population"] > 0),
    town_df["federalShareObligated"] / town_df["total_population"],
    np.nan,
)
# makes it easier to visualize and analyze, given the skewness of funding distribution and many towns with 0 funding
town_df["log_funding_per_capita"] = np.log1p(town_df["funding_per_capita"])

# calculate funding per occupied housing unit and log of funding per occupied housing unit
town_df["funding_per_occupied_unit"] = np.where(
    (town_df["occupied_housing_units"] > 0),
    town_df["federalShareObligated"] / town_df["occupied_housing_units"],
    np.nan,
)

# calculate log of occupied housing units
town_df["log_funding_per_occupied_unit"] = np.log1p(
    town_df["funding_per_occupied_unit"]
)

# boolean flag for towns with funding, may help with analysis
town_df["has_funding"] = town_df["funding_per_capita"] > 0

town_df.shape

(257, 37)

In [26]:
town_df["has_funding"].value_counts()

has_funding
False    137
True     120
Name: count, dtype: int64

In [27]:
# check distribution of funding per capita for towns with funding, to understand skewness and inform analysis approach
town_df[town_df["funding_per_capita"] > 0]["funding_per_capita"].describe()

count     120.000000
mean      241.398625
std       369.061391
min         0.655172
25%        22.085446
50%       106.590387
75%       289.213203
max      2110.756106
Name: funding_per_capita, dtype: float64

In [28]:
# check how the log transformation has affected the distribution, to confirm it's more normalized and appropriate for analysis
town_df[town_df["funding_per_capita"] > 0]["log_funding_per_capita"].describe()

count    120.000000
mean       4.453672
std        1.630633
min        0.503905
25%        3.139131
50%        4.678330
75%        5.670597
max        7.655275
Name: log_funding_per_capita, dtype: float64

In [29]:
# check distribution of funds per occupied housing unit for towns with funding, to understand skewness and inform analysis approach
town_df[town_df["funding_per_occupied_unit"] > 0][
    "funding_per_occupied_unit"
].describe()

count     120.000000
mean      554.803694
std       828.032042
min         1.563591
25%        62.769130
50%       228.116349
75%       635.391342
max      4739.515074
Name: funding_per_occupied_unit, dtype: float64

In [30]:
# check log transformation of funds per occupied housing unit
town_df[town_df["funding_per_occupied_unit"] > 0][
    "log_funding_per_occupied_unit"
].describe()

count    120.000000
mean       5.280490
std        1.656173
min        0.941409
25%        4.155266
50%        5.432993
75%        6.455009
max        8.463901
Name: log_funding_per_occupied_unit, dtype: float64

In [31]:
# split off state row
statewide_row = town_df[town_df["town_name"] == "State, Regional, or Unknown"]
town_df_analysis = town_df[town_df["town_name"] != "State, Regional, or Unknown"].copy()

# display and export
print("State, Regional, or Unknown row:")
display(statewide_row)
statewide_row.to_csv("../data/cleaned/fema_hma_non_town_level.csv", index=False)

# display, alphabetize, and export
print("Town-level data for analysis:")
print(town_df_analysis.shape)
display(town_df_analysis.head())
town_df_analysis = town_df_analysis.sort_values("town_name")
town_df_analysis.to_csv("../data/cleaned/fema_hma_town_level.csv", index=False)

State, Regional, or Unknown row:


,GEOID,town_name,total_population,occupied_housing_units,funding_acquisition_buyouts,funding_admin,funding_ecosystem_restoration,funding_equipment,funding_flood_control,funding_infrastructure_protection,...,share_infrastructure_protection,share_infrastructure_stormwater,share_other,share_planning,share_structure_protection,funding_per_capita,log_funding_per_capita,funding_per_occupied_unit,log_funding_per_occupied_unit,has_funding
206,NaN,"State, Regional, or Unknown",NaN,NaN,35247963.81,0.0,3707273.76,0.0,0.0,1682801.83,...,0.040881,0.012766,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False


Town-level data for analysis:
(256, 37)


,GEOID,town_name,total_population,occupied_housing_units,funding_acquisition_buyouts,funding_admin,funding_ecosystem_restoration,funding_equipment,funding_flood_control,funding_infrastructure_protection,...,share_infrastructure_protection,share_infrastructure_stormwater,share_other,share_planning,share_structure_protection,funding_per_capita,log_funding_per_capita,funding_per_occupied_unit,log_funding_per_occupied_unit,has_funding
0,5.000100e+09,Addison town,1175.0,490.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,5.001900e+09,Albany town,934.0,394.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,5.001301e+09,Alburgh town,1832.0,786.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,5.002701e+09,Andover town,616.0,238.0,106817.55,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,173.405114,5.161381,448.813235,6.108832,True
4,5.000301e+09,Arlington town,2598.0,864.0,0.00,0.0,0.0,0.0,3125.0,3125.0,...,0.333333,0.0,0.0,0.0,0.333333,3.608545,1.527912,10.850694,2.472386,True
